In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import itertools


df = pd.read_excel('2.1阳光信息.xlsx')
df['权重和']=df['权重和']/10000
df = df.drop('flag',axis=1)
# df
df.columns

Index(['Y', '路障', '撑杆', '铁桶', '读报', '铁门', '橄榄', '舞王', '潜水', '冰车', '海豚', '小丑',
       '气球', '矿工', '跳跳', '蹦极', '扶梯', '投篮', '白眼', '红眼', '权重和'],
      dtype='object')

In [2]:
# handle
df['Y'][:10] -= 65.52
df['Y'][10:] -= 37.17

C:\Users\Administrator\AppData\Local\Temp\ipykernel_1324\3292867118.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['Y'][:10] -= 65.52
C:\Users\Administrator\AppData\Local\Temp\ipykernel_1324\3292867118.py:2: SettingWithCopyWarning: 


In [3]:
# +- handle
cols =  df.columns.values
cols = cols[~np.isin(cols, ['Y', '权重和'])]
# df[cols] = df[cols].replace(0,-1)

In [4]:
def OLSmodel(df,const=True):
    X = df.iloc[:,1:]
    y = -df['Y']
    X_sm = sm.add_constant(X) if const else X
    model_sm = sm.OLS(y, X_sm).fit()
    print(model_sm.summary())
    return pd.DataFrame(model_sm.params)

In [5]:
def OLS_factor_model(df,ord=2):
    df['Y'] = -df['Y']
    
    formula = 'Y ~ '
    df_cols = df.columns[~np.isin(df.columns, ['Y', '权重和'])]

    formula_terms = []
    for order in range(1, ord+1):  # 1阶、2阶、3阶
        for combo in itertools.combinations(df_cols, order):
            formula_terms.append(':'.join(combo))

    formula = formula+'+'.join(formula_terms)
    model = ols(formula, data=df).fit()
    print(model.summary())



In [6]:
def OLS_factor_DIY_model(df,main_factors,cross_factors):
    df['Y'] = -df['Y']
    
    formula = 'Y ~ '
    df_cols = df.columns[~np.isin(df.columns, ['Y', '权重和'])]

    formula_terms = []
            
    for fact in main_factors:
        formula_terms.append(fact)
    
    for fact in cross_factors:
        formula_terms.append(fact)
        
    formula = formula+'+'.join(formula_terms)
    print(formula)
    model = ols(formula, data=df).fit()
    print(model.summary())
    
    # b 方差分析 及 输出 ANOVA 表和显著效应
    anova_table = sm.stats.anova_lm(model, typ=2,return_type='DataFrame')
    print("ANOVA 表：")
    print(anova_table)
    print(f"Total     {model.centered_tss:.6f} ")

    significant_effects = anova_table[anova_table['PR(>F)'] < 0.05].index
    print("\n显著效应(p < 0.05):")
    print(significant_effects)


In [7]:
# ['Y','撑杆','铁桶','铁门','橄榄','舞王','冰车',
# '小丑','气球','矿工','跳跳','蹦极','扶梯','投篮','白眼','红眼']
OLS_factor_model(df[[
        'Y','撑杆','铁桶','铁门','橄榄','舞王','冰车',
        '小丑','气球','矿工','蹦极','投篮','白眼','红眼'
    ]],
    ord=2
)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Tue, 27 May 2025   Prob (F-statistic):                nan
Time:                        05:56:08   Log-Likelihood:                 850.22
No. Observations:                  35   AIC:                            -1630.
Df Residuals:                       0   BIC:                            -1576.
Df Model:                          34                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -193.3561        inf         -0        n

C:\Users\Administrator\AppData\Local\Temp\ipykernel_1324\3926916747.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Y'] = -df['Y']
c:\MyFile\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
c:\MyFile\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
c:\MyFile\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


In [8]:
# ['Y','撑杆','铁桶','铁门','橄榄','舞王','冰车',
# '小丑','气球','矿工','跳跳','蹦极','扶梯','投篮','白眼','红眼']
OLS_factor_DIY_model(df,
    ['气球','矿工',     '蹦极',
     
     '红眼','橄榄',
     
     '铁桶','铁门','舞王','冰车',
    ],
    ['红眼:橄榄','投篮:红眼','白眼:红眼','投篮:白眼','红眼:冰车']
)

Y ~ 气球+矿工+蹦极+红眼+橄榄+铁桶+铁门+舞王+冰车+红眼:橄榄+投篮:红眼+白眼:红眼+投篮:白眼+红眼:冰车
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.924
Model:                            OLS   Adj. R-squared:                  0.871
Method:                 Least Squares   F-statistic:                     17.41
Date:                Tue, 27 May 2025   Prob (F-statistic):           3.28e-08
Time:                        05:56:08   Log-Likelihood:                -236.21
No. Observations:                  35   AIC:                             502.4
Df Residuals:                      20   BIC:                             525.7
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------

In [9]:
# sample all
OLSmodel(df[['Y','撑杆','铁桶','铁门','橄榄','舞王','冰车','气球','矿工','蹦极','投篮','白眼','红眼']])
# OLSmodel(df[['Y',    '橄榄', '舞王',  '冰车',  '气球', '矿工',  '红眼', '权重和']],False)


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.924
Model:                            OLS   Adj. R-squared:                  0.882
Method:                 Least Squares   F-statistic:                     22.27
Date:                Tue, 27 May 2025   Prob (F-statistic):           1.50e-09
Time:                        05:56:08   Log-Likelihood:                -236.26
No. Observations:                  35   AIC:                             498.5
Df Residuals:                      22   BIC:                             518.7
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1401.4782    369.514      3.793      0.0

,0
const,1401.478164
撑杆,-6.424634
铁桶,336.824304
铁门,424.815338
橄榄,-596.867709
舞王,-373.754209
冰车,-336.651194
气球,-925.084441
矿工,-420.493494
蹦极,-165.657456
